In [ ]:
# Copyright (c) 2024 Zenteiq Aitech Innovations Private Limited and
# AiREX Lab, Indian Institute of Science, Bangalore.
# All rights reserved.
#
# This file is part of SciREX
# (Scientific Research and Engineering eXcellence Platform),
# developed jointly by Zenteiq Aitech Innovations and AiREX Lab
# under the guidance of Prof. Sashikumaar Ganesan.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#
# For any clarifications or special considerations,
# please contact: contact@scirex.org

import jax
import jax.numpy as jnp
from flax import linen as nn
from typing import Tuple, Optional
import itertools

class SpectralConvDerv(nn.Module):
    """
    N-dimensional Spectral Convolution layer (supports 1D, 2D, 3D, and beyond).
    
    The dimensionality is automatically inferred from the length of `n_modes`.
    
    This layer performs a convolution in the Fourier domain by:
    1. Transforming the input to the frequency domain using a Real FFT (RFFT).
    2. Multiplying the lower Fourier modes by learnable complex weights.
    3. Inverse transforming the filtered signal back to the spatial domain.
    
    Attributes:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        n_modes (Tuple[int, ...]): Number of Fourier modes to retain for each spatial dimension.
        init_std (float, optional): Standard deviation for weight initialization.
    """
    in_channels: int
    out_channels: int
    n_modes: Tuple[int, ...]
    init_std: Optional[float] = None
    deriv_order: int = 1              # 1 for first derivative, 2 for second

    @nn.compact
    def __call__(self, x: jnp.ndarray, derv_order:int) -> jnp.ndarray:
        # x shape: (batch, dim1, dim2, ..., dimN, in_channels)
        n_dim = len(self.n_modes)
        batch = x.shape[0]
        spatial_dims = x.shape[1:-1]
        
        # 0. Safety Check: Ensure requested modes don't exceed Nyquist limits
        for i, (mode, dim) in enumerate(zip(self.n_modes, spatial_dims)):
            # The last dimension is roughly halved due to RFFT
            max_modes = dim // 2 + 1 if i == n_dim - 1 else dim
            if mode > max_modes:
                raise ValueError(
                    f"n_modes[{i}]={mode} exceeds maximum allowed modes ({max_modes}) "
                    f"for spatial dimension size {dim}."
                )

        # 1. FFT
        axes = tuple(range(1, n_dim + 1))
        x_ft = jnp.fft.rfftn(x, axes=axes, norm="ortho")
        
        # 2. Weights Initialization
        scale = 0.05 if self.init_std is None else self.init_std
        weights_shape = (self.in_channels, self.out_channels) + self.n_modes
        
        # For N dimensions, the number of corners in frequency space is 2**(N-1).
        n_corners = 2**(n_dim - 1)
        weights = [
            self.param(f'weights_{i+1}', jax.nn.initializers.normal(stddev=scale), weights_shape, jnp.complex64)
            for i in range(n_corners)
        ]
        
        # Create output tensor in frequency domain
        out_ft_shape = (batch,) + spatial_dims[:-1] + (spatial_dims[-1] // 2 + 1,) + (self.out_channels,)
        out_ft = jnp.zeros(out_ft_shape, dtype=jnp.complex64)
        
        # 3. Build dynamic einsum string (safely avoids hardcoded letter limits)
        # We reserve 'b' for batch, 'i' for in_channels, 'o' for out_channels
        available_letters = "acdefghjklmnpqrstuvwxyz"
        if n_dim > len(available_letters):
             raise ValueError(f"Too many spatial dimensions ({n_dim}) for einsum string generation.")
        spatial_letters = available_letters[:n_dim]
        einsum_str = f"b{spatial_letters}i,io{spatial_letters}->b{spatial_letters}o"
        
        # 4. Apply Weights to Frequency Corners
        corner_idx = 0
        for signs in itertools.product([1, -1], repeat=n_dim - 1):
            slices_in = [slice(None)]  # batch
            slices_out = [slice(None)] # batch
            
            for d, sign in enumerate(signs):
                modes = self.n_modes[d]
                if sign == 1:
                    slices_in.append(slice(None, modes))
                    slices_out.append(slice(None, modes))
                else:
                    slices_in.append(slice(-modes, None))
                    slices_out.append(slice(-modes, None))
                    
            # Last spatial dimension (always positive frequencies for RFFT)
            last_modes = self.n_modes[-1]
            slices_in.append(slice(None, last_modes))
            slices_out.append(slice(None, last_modes))
            
            # Channel dimension
            slices_in.append(slice(None))
            slices_out.append(slice(None))
            
            # Extract corner, multiply, and inject back
            x_corner = x_ft[tuple(slices_in)]
            w_corner = weights[corner_idx]
            
            out_corner = jnp.einsum(einsum_str, x_corner, w_corner)
            out_ft = out_ft.at[tuple(slices_out)].set(out_corner)
            
            corner_idx += 1
            
        # ---------------------------------------------------------------------
        # 5 Compute the Spatial Derivative (i * k multiplier)
        # ---------------------------------------------------------------------
        nx,ny = x.shape[1], x.shape[2] 

        """"Above expression is incorrect because it assumes that the input to this function
        is the input to the model, rather the output of the last FNO block."""

        Lx,Ly = 1,1 # Assuming 2D spatial dimensions for derivative computation
        k_x = jnp.fft.fftfreq(nx) * nx
        k_broadcast_x = ((2.0 * jnp.pi / Lx) * k_x).reshape((1, nx, 1, 1))
        mult_x = (1j * k_broadcast_x) ** self.deriv_order
        
        # d/dy multiplier
        k_y = jnp.fft.rfftfreq(ny) * ny
        k_broadcast_y = ((2.0 * jnp.pi / Ly) * k_y).reshape((1, 1, ny // 2 + 1, 1))
        mult_y = (1j * k_broadcast_y) ** self.deriv_order
        
        # Branch the Fourier tensor
        out_ft_x = out_ft * mult_x
        out_ft_y = out_ft * mult_y
        
        # ---------------------------------------------------------------------
        # 5. Inverse 2D FFT on both branches
        # ---------------------------------------------------------------------
        dv_dx = jnp.fft.irfftn(out_ft_x, s=(nx, ny), axes=(1, 2), norm="ortho")
        dv_dy = jnp.fft.irfftn(out_ft_y, s=(nx, ny), axes=(1, 2), norm="ortho")
        
        # Stack along a new final axis
        # Resulting shape: (batch_size, nx, ny, channels, 2)
        return jnp.stack([dv_dx, dv_dy], axis=-1)

In [ ]:
import jax
import jax.numpy as jnp


class loss_pde_FNO:

    def __call__(self, physical_system, config,params, state, batch):
       
       u_p_jacobian = jnp.einsum('bxyc, bxyk -> ikl', layer_derv.first_derv_projection(params,state,v_L),SpectralConvDerv(params,state,batch))
    #    u_pp_jacobian = jnp.einsum('ij, jkl -> ikl', second_derv_projection(params,state,batch),second_derivative_SpectralConvDerv(params,state,batch))


       if  physical_system == 'Poisson2D':
            # Poisson equation: Δu = f
            # Here, we assume f is given by the batch (or can be computed from the batch)
            """f = batch['f']  # Assuming f is part of the batch
            residual = u_pp_jacobian - f  # Δu - f"""
            loss = jnp.mean(residual**2)  # Mean squared error of the residual
            return loss
        

In [ ]:
import jax
import jax.numpy as jnp

class layer_derv():
    def __init__(self, params, state, batch):
        self.params = params
        self.state = state
        self.batch = batch
        
    projection_params = self.state.params['projection_layer']

    # Unpack the parameters for the first layer of the MLP
    # Note: If scirex uses Conv layers instead of Dense, these might be 'Conv_0' and 'Conv_1'
    W1 = projection_params['dense_0']['kernel']
    b1 = projection_params['dense_0']['bias']

    # Unpack the parameters for the second layer
    W2 = projection_params['dense_1']['kernel']
    b2 = projection_params['dense_1']['bias']

    projection_layer = lambda x_point: jnp.dot(nn.gelu(jnp.dot(x_point, W1) + b1), W2) + b2

    def first_derv_projection(self):
        """
        v_L: Standard output of Fourier block (batch, nx, ny, C_in)
        v_L_prime: Derivative output of Fourier block (batch, nx, ny, C_in)
        """
        # 2. Get the Jacobian function for a single point
        jac_fn = jax.jacfwd(projection_layer)

        # 3. Vectorize over ny, nx, and batch
        vmap_ny = jax.vmap(jac_fn, in_axes=0)
        vmap_nx = jax.vmap(vmap_ny, in_axes=0)
        vmap_batch = jax.vmap(vmap_nx, in_axes=0)

        # 4. Compute Q'(v_L)
        # Shape will be: (batch, nx, ny, C_out, C_in)
        Q_prime = vmap_batch(v_L) 

        return Q_prime

    def second_derv_projection(self):
        # Compute the second derivative of the projection layer w.r.t. input x
        return jax.grad(self.first_derv_projection, argnums=4)(self.params['W1'], self.params['W2'], self.params['b1'], self.params['b2'], self.batch['x'])

In [1]:
# Assume Q_prime is the output of first_derv_projection()
# Shape: (batch_size, nx, ny, C_out, H)
Q_prime = layer_derv_instance.first_derv_projection()

# Assume v_prime is the output of SpectralConvDerv
# Shape: (batch_size, nx, ny, H, 2)
v_prime = spectral_conv_derv_instance(x, derv_order=1)

# Contract over the latent channel dimension 'H' (represented by 'i')
# b = batch, x = nx, y = ny, o = C_out, i = H, d = 2 (spatial derivatives)
u_prime = jnp.einsum('bxyoi, bxyid -> bxyod', Q_prime, v_prime)

# Final u_prime shape: (batch_size, nx, ny, C_out, 2)

NameError: name 'layer_derv_instance' is not defined

In [ ]:
def second_derv_projection(self, v_L):
        """
        Computes Q''(v_L) - The Hessian of the projection layer w.r.t its input.
        """
        # 1. Double jacfwd to get the Hessian for a single point
        # jacfwd is usually preferred over jacrev for tall/square matrices
        hessian_fn = jax.jacfwd(jax.jacfwd(self.projection_layer))

        # 2. Vectorize over ny, nx, and batch
        vmap_ny = jax.vmap(hessian_fn, in_axes=0)
        vmap_nx = jax.vmap(vmap_ny, in_axes=0)
        vmap_batch = jax.vmap(vmap_nx, in_axes=0)

        # 3. Compute Q''(v_L)
        # Assuming v_L shape is (batch, nx, ny, H) and projection outputs C_out
        # Output Shape will be: (batch, nx, ny, C_out, H, H)
        Q_prime2 = vmap_batch(v_L) 

        return Q_prime2

In [ ]:
# Term 1: v_L'^T * Q''(v_L) * v_L'
# We contract the two H dimensions (i and j) of the Hessian with our two v_prime vectors.
# -------------------------------------------------------------------
term_1 = jnp.einsum(
    'bxyoij, bxyi, bxyj -> bxyo', 
    Q_prime2, v_prime_x, v_prime_x
)

# -------------------------------------------------------------------
# Term 2: Q'(v_L) * v_L''
# We contract the single H dimension (i) of the Jacobian with the second derivative vector.
# -------------------------------------------------------------------
term_2 = jnp.einsum(
    'bxyoi, bxyi -> bxyo', 
    Q_prime, v_prime2_x
)

# -------------------------------------------------------------------
# Final Second Derivative: d^2u / dx^2
# Shape: (batch, nx, ny, C_out)
# -------------------------------------------------------------------
d2u_dx2 = term_1 + term_2